In [6]:
import os
import re
import pandas as pd
from bs4 import BeautifulSoup

# --- 1. HÀM CHUYỂN SỐ HÁN SIÊU CẤP (Xử lý cả kiểu 一〇 và 十) ---
def han_to_arabic(s):
    d = {'一':1,'二':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9,'十':10,'百':100,'〇':0,'零':0}
    if not s: return 0
    # Kiểu 1: Một-Không (一〇 = 10, 一一 = 11, 二二二 = 222)
    if all(c in d for c in s) and (len(s) > 1 and '十' not in s and '百' not in s):
        res = 0
        for char in s: res = res * 10 + d[char]
        return res
    # Kiểu 2: Văn bản (十 = 10, 二十二 = 22)
    res, tmp = 0, 0
    for char in s:
        v = d.get(char, 0)
        if v == 10:
            res += (tmp if tmp > 0 else 1) * 10
            tmp = 0
        elif v == 100:
            res += (tmp if tmp > 0 else 1) * 100
            tmp = 0
        else: tmp = v
    return res + tmp

# --- 2. LÀM SẠCH RÁC HỆ THỐNG ---
def clean_system_garbage(text):
    if not text: return ""
    text = re.sub(r'\[[a-zA-Z0-9]+\]', '', text) # Xóa [1]
    # Xóa mã hiệu trang CBETA cực mạnh
    text = re.sub(r'[A-Z0-9]{2,}n[0-9]{2,}.*p[0-9]{2,}[a-z][0-9]{2,}', '', text)
    text = re.sub(r'p[0-9]{4}[a-z][0-9]{2}', '', text)
    text = re.sub(r'QC[0-9]+n[0-9]+_p[0-9]+[a-z][0-9]+', '', text)
    # Xóa dòng chú thích dị bản
    if sum(1 for ed in ["【大】", "【宋】", "【元】", "【明】", "【麗】"] if ed in text) >= 2:
        return ""
    return text.strip()

# --- 3. QUY TRÌNH XỬ LÝ CHÍNH ---
all_data = []
# Tìm mọi dấu ngoặc chứa số Hán tự
sutta_marker = re.compile(r'[（\(]([一二三四五六七八九十百〇零]+)[）\)]')

input_root = "html_input"
all_files = sorted([os.path.join(r, f) for r, d, fs in os.walk(input_root) for f in fs if f.lower().endswith(('.html', '.htm'))])

print(f"📂 Đang bắt đầu xử lý {len(all_files)} file...")

global_curr_no = 0

for path in all_files:
    with open(path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    # Lấy text theo khối (p, div) để tránh dính chữ
    blocks = soup.find_all(['p', 'div'])
    para_idx = 1

    for block in blocks:
        line_raw = block.get_text().strip()
        if len(line_raw) < 5: continue

        # Thử tìm số kinh trong dòng này
        match = sutta_marker.search(line_raw)
        if match:
            num_part = match.group(1)
            new_no = han_to_arabic(num_part)

            # Nếu tìm thấy số kinh hợp lý (1-222) và gần với số hiện tại
            if 0 < new_no <= 222:
                # Nếu số mới lớn hơn số cũ, hoặc là bài 1
                if new_no > global_curr_no or (new_no == 1 and global_curr_no == 0):
                    global_curr_no = new_no
                    para_idx = 1
                    #print(f"  -> Đã tìm thấy kinh số: {global_curr_no}")
                    continue

        # Nếu đã xác định được đang ở trong một bài kinh
        if global_curr_no > 0:
            cleaned = clean_system_garbage(line_raw)
            if len(cleaned) > 10 and "品第" not in cleaned and "卷第" not in cleaned:
                # Chỉ giữ Hán tự cho cột Cleaned
                text_only = re.sub(r'[^\u4e00-\u9fff]', '', cleaned)
                if len(text_only) > 5:
                    all_data.append({
                        "Sutta_No": global_curr_no,
                        "Segment_ID": f"MA.{global_curr_no}.{para_idx}",
                        "Content": cleaned,
                        "Cleaned_Content": text_only
                    })
                    para_idx += 1

# 4. KIỂM TRA VÀ XUẤT FILE
if not all_data:
    print("❌ Không bóc tách được dữ liệu. Hãy kiểm tra định dạng Zip.")
else:
    df_ma = pd.DataFrame(all_data).drop_duplicates(subset=['Segment_ID'])
    output_name = "Hantu_MA_222_Full_Success.csv"
    df_ma.to_csv(output_name, index=False, encoding='utf-8-sig')

    print("\n" + "="*30)
    print(f"📊 BÁO CÁO CUỐI CÙNG:")
    found_suttas = sorted(df_ma['Sutta_No'].unique())
    print(f"✅ Số bài kinh tìm thấy: {len(found_suttas)}/222")
    missing = [i for i in range(1, 223) if i not in found_suttas]
    if missing:
        print(f"⚠ Các bài còn thiếu: {missing}")
    else:
        print("🎉 HOÀN HẢO! Đã tìm đủ 222 bài.")
    print(f"📝 Tổng số dòng: {len(df_ma)}")
    print("="*30)
    print(f"File đã lưu: {output_name}")

📂 Đang bắt đầu xử lý 60 file...

📊 BÁO CÁO CUỐI CÙNG:
✅ Số bài kinh tìm thấy: 221/222
⚠ Các bài còn thiếu: [48]
📝 Tổng số dòng: 8617
File đã lưu: Hantu_MA_222_Full_Success.csv
